In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# unit and sanity checks!
"""--------------------------------------------"""
# add pooled r2 -> rerun movement r2 nb
# replicate neurotheory plots
# - clean up

# > check the fits for different regularization constants
# define responsive

# one regressor
# cvr2 with strategy model params
# add time
"""--------------------------------------------"""

## init

In [ ]:
# check that nans are still caught as different between mb/mf with out of pool averging

In [ ]:
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id)
encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

In [ ]:
encoder.verify()
# encoder_mb.verify()
encoder_mf.verify()

In [ ]:
encoder.view_fits()

In [ ]:
encoder.view_weights()

In [ ]:
# se = ShuffledEncoder(
#     subj_id,
#     sess_id,
#     tv_keys=[
#         "response",
#         "rewarded",
#         "block_side",
#         "strategy",
#         "response_prev",
#         "rewarded_prev",
#     ],
# )
# se.plot_cvr2()
# se.plot_dr2()|
# se.plot_bound_r2()

## r2 comp between regions and strategies 

scatter version is in .verify()

In [ ]:
# r2 between DMS and DLS
from core.viz import plot_kdes

# scores
scores = {
    f"{reg}, {model}": encoder.scores[model][encoder.reg_idxs[reg]]
    for reg in encoder.regions
    for model in ["baseline", "encoder"]
}
# scores_mb = {f"{reg}, {model}": encoder_mb.scores[model][encoder_mb.reg_idxs[reg]] for reg in encoder_mb.regions for model in ['baseline', 'encoder']}
scores_mf = {
    f"{reg}, {model}": encoder_mf.scores[model][encoder_mf.reg_idxs[reg]]
    for reg in encoder_mf.regions
    for model in ["baseline", "encoder"]
}

# styles
linestyles = {"baseline": "--", "encoder": "-"}
colors = {"DMS": "#562E9C", "DLS": "#009D51"}
styles = {
    f"{reg}, {model}": {"linestyle": linestyles[model], "color": colors[reg]}
    for reg in encoder.regions
    for model in ["baseline", "encoder"]
}

plot_kdes(scores, label=r"$r^2$", xlim=(-0.25, 1), add_means=False, line_kwargs=styles)
plot_kdes(
    scores_mf, label=r"$r^2$, mf", xlim=(-0.25, 1), add_means=False, line_kwargs=styles
)
# plot_kdes({f"{reg}, {model}": encoder_mb.scores[model][encoder_mb.reg_idxs[reg]] for reg in encoder_mb.regions for model in ['baseline', 'encoder']})

## weight comp between regions and strategies

### kde

In [ ]:
from core.viz import plot_kdes

# iterate through all regressors and save
regressor = "response_left"

# weights
weights_strategy = {
    k: {
        f"{reg}": encoder_.encoder_weights[
            encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
        ]
        for reg in encoder_.regions
    }
    for k, encoder_ in {"both": encoder, "mf": encoder_mf}.items()
}

weights_reg = {
    reg: {
        f"{k}": encoder_.encoder_weights[
            encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
        ]
        for k, encoder_ in {"both": encoder, "mf": encoder_mf}.items()
    }
    for reg in encoder.regions
}
# styles

colors = {"DMS": "#562E9C", "DLS": "#009D51"}
styles = {f"{reg}": {"color": colors[reg]} for reg in encoder.regions}

for weights in [weights_strategy, weights_reg]:
    for k in weights:
        plot_kdes(
            weights[k],
            label=rf"$\beta$ {regressor}, {k}",
            add_means=False,
            line_kwargs=styles,
        )

### scatter, hist 2d, contour

In [ ]:
from core.viz import plot_scatter

plot_scatter(
    weights_strategy["both"]["DLS"],
    weights_strategy["mf"]["DLS"],
    xlabel=rf"$\beta$ {regressor}, both",
    ylabel=rf"$\beta$ {regressor}, mf",
    add_unity=True,
)

In [ ]:
plt.figure()
plt.hist2d(
    weights_strategy["both"]["DLS"],
    weights_strategy["mf"]["DLS"],
    range=[[-20, 20], [-20, 20]],
    bins=50,
    cmap="Blues",
    density=True,
)
plt.colorbar()
plt.show()

In [ ]:
import seaborn as sns

x = weights_strategy["both"]["DLS"]
y = weights_strategy["mf"]["DLS"]

plt.figure()
sns.kdeplot(x=x, y=y, cmap="Blues", levels=20, thresh=0, clip=((-20, 20), (-20, 20)))
plt.show()